<a id="inicio"></a>

# Electricity Demand Monitoring with InfluxDB — Part 2: Forecasting

## Objective

Build a day-ahead hourly forecaster for Spanish electricity demand. Train a [Prophet](https://facebook.github.io/prophet/) model on the ingested history from Part 1, evaluate it daily across a held-out window, and write the forecast back into InfluxDB for dashboard display alongside the actual demand.

## Methodology

- **Training data**: Hourly `RealDemand` series pulled from the `capstone7` bucket
- **Model**: Prophet via [Darts](https://github.com/unit8co/darts), a unified Python forecasting library
- **Evaluation protocol**: Rolling-origin — for each evaluation day, train on all history up to that day and forecast 24 hours ahead
- **Deployment**: Write the forecast back to InfluxDB under a new field `CP7Forecast` so the existing dashboard can overlay it

<div class="alert alert-block alert-info">

### ℹ️ Note on cell outputs

The cell outputs visible in this notebook were captured during the **original run** and are shown here as a visual reference for portfolio review. Some artifacts may still appear in Spanish because the outputs predate the English code translation.

**To re-run locally:**

1. **Prerequisite**: run Part 1 first so your InfluxDB instance has ingested data to train on.
2. With the same Docker stack running, install Python dependencies:
   `pip install -r requirements.txt`
3. The notebook trains a [Darts](https://github.com/unit8co/darts) Prophet model on hourly demand data and writes a day-ahead forecast back to InfluxDB under the field name `CP7Forecast`.
4. Launch Jupyter, open this file, and run top-to-bottom.

</div>

---

---

<a id="indice"></a>
## Contents

* [1. Forecasting Model](#section1)
* [2. Forecast Ingestion and Visualization](#section2)

---

In [1]:
import matplotlib.pyplot as plt
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
import pandas as pd
import numpy as np

# Optimiza los gráficos para pantalla retina
%config InlineBackend.figure_format = 'retina'
# Por defecto usamos el backend inline
%matplotlib inline

# Oculta warnings
import warnings
warnings.simplefilter('ignore')

# La libreta ocupa así el 95% de la pantalla
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

import logging
logger = logging.getLogger('cmdstanpy')
logger.addHandler(logging.NullHandler())
logger.propagate = False
logger.setLevel(logging.CRITICAL)

In this part we build a forecasting model from the data ingested into InfluxDB in Part 1, then write day-ahead forecasts back into the same series so the dashboard can display them. The model trains on all data up to the most recent complete day; the next day's forecast is served regardless of whether real data is already available (if it is, we ignore it during training).

---

<a id="section1"></a>
## 1. Forecasting Model

First we pull the training data from InfluxDB.

---

### Task 1: Load data from InfluxDB

Implement a function that returns a DataFrame with every available `RealDemand` observation from your InfluxDB instance. We only need the actual demand — filter out the REE forecast using Flux.

Provide your InfluxDB connection info (the defaults match the Docker stack used in Part 1).

*Hint*: The Flux query you built for the Part 1 dashboard is a good starting point.

In [2]:
token = "mcidaen_token"
org = "mcidaen"
bucket = "capstone7"

client = InfluxDBClient(url="http://influxdb:8086", token=token)

In [4]:
def load_from_influxdb(client: InfluxDBClient, bucket: str, org: str) -> pd.DataFrame:
    """
    Devuelve un DataFrame con la serie temporal de RealDemand desde InfluxDB.
    """
    query = f'''
    from(bucket: "{bucket}")
      |> range(start: 0)
      |> filter(fn: (r) => r["_measurement"] == "demand")
      |> filter(fn: (r) => r["_field"] == "RealDemand")
      |> keep(columns: ["_time", "_value"])
    '''

    query_api = client.query_api()
    tables = query_api.query_data_frame(query=query, org=org)

    if isinstance(tables, list):  # En caso de múltiples tablas (puede pasar con algunos datos)
        df = pd.concat(tables, ignore_index=True)
    else:
        df = tables

    df_query = df.loc[:, ['_time', '_value']]
    df_query['datetime'] = pd.to_datetime(df_query['_time'], utc=True).dt.tz_localize(None)
    df_query['RealDemand'] = df_query['_value']
    df_query = df_query[['datetime', 'RealDemand']]
    df_query.index = df_query['datetime']
    return df_query['RealDemand'].sort_index()


In [5]:
df = load_from_influxdb(client, bucket, org)
df['2024'].sample(20, random_state=0).sort_index()

datetime
2024-01-14 05:00:00    21912.916667
2024-01-23 13:00:00    32208.833333
2024-02-22 22:00:00    28256.833333
2024-03-13 05:00:00    27400.250000
2024-04-09 02:00:00    21497.750000
2024-04-17 10:00:00    27627.500000
2024-06-03 02:00:00    21675.666667
2024-06-11 19:00:00    31106.000000
2024-06-15 05:00:00    22771.166667
2024-06-24 02:00:00    21709.416667
2024-08-16 11:00:00    28886.583333
2024-09-08 04:00:00    21697.666667
2024-09-21 17:00:00    26574.666667
2024-09-24 04:00:00    24955.250000
2024-09-30 08:00:00    29362.250000
2024-10-01 10:00:00    30193.250000
2024-10-07 20:00:00    30029.333333
2024-11-16 07:00:00    25059.250000
2024-12-14 12:00:00    29838.916667
2024-12-22 15:00:00    27064.000000
Name: RealDemand, dtype: float64

In [7]:
df_formatted = df['2024'].sample(20, random_state=0).sort_index().reset_index()
df_formatted.columns = ['datetime', 'RealDemand']
display(df_formatted)


,datetime,RealDemand
0,2024-01-14 05:00:00,21912.916667
1,2024-01-23 13:00:00,32208.833333
2,2024-02-22 22:00:00,28256.833333
3,2024-03-13 05:00:00,27400.250000
4,2024-04-09 02:00:00,21497.750000
5,2024-04-17 10:00:00,27627.500000
6,2024-06-03 02:00:00,21675.666667
7,2024-06-11 19:00:00,31106.000000
8,2024-06-15 05:00:00,22771.166667
9,2024-06-24 02:00:00,21709.416667


Expected result (your dataset may differ):

| datetime            |   RealDemand |
|:--------------------|-------------:|
| 2024-01-02 07:00:00 |      28070.3 |
| 2024-01-07 06:00:00 |      20867.5 |
| 2024-01-13 11:00:00 |      29865.4 |
| 2024-01-13 18:00:00 |      31708.2 |
| ...

Next, build a forecaster. You can use any model that integrates with the rest of the stack — we chose Prophet (via Darts).

We wrap training and day-ahead prediction in a single function so we can evaluate it on multiple days, simulating production-style rolling forecasts.

---

### Task 2: Forecasting function

Implement a function that takes:
- `training` — historical demand as a `darts.TimeSeries`
- `test_hours` — number of hours to forecast

and returns a forecast `TimeSeries` named `CP7Forecast`. Internally, fit a Darts model on `training`, then predict `test_hours` steps ahead.

*Optional*: Try alternative models / techniques from the module and compare forecasting performance.

In [11]:
import datetime
from darts.models import Prophet
import darts
from darts import TimeSeries

def train_forecast(training: darts.TimeSeries, steps: int) -> darts.TimeSeries:

    model = Prophet()
    model.fit(training)
    forecast = model.predict(steps)
    forecast = forecast.with_columns_renamed(forecast.columns[0], "CP7Forecast")
    return forecast


In [9]:
training = darts.TimeSeries.from_dataframe(pd.DataFrame(df.loc[:datetime.datetime(2024, 2, 1),]))

In [10]:
result = train_forecast(training, 24)
result.pd_dataframe()

component,CP7Forecast
datetime,
2024-02-01 01:00:00,23878.005719
2024-02-01 02:00:00,22972.403732
2024-02-01 03:00:00,22651.954196
2024-02-01 04:00:00,23512.298712
2024-02-01 05:00:00,25743.387311
2024-02-01 06:00:00,28702.006133
2024-02-01 07:00:00,31239.705365
2024-02-01 08:00:00,32524.010106
2024-02-01 09:00:00,32597.765308


Expected result (yours may differ):

| datetime            |  CP7Forecast |
|:--------------------|-------------:|
| 2024-02-01 01:00:00 |      25338.6 |
| 2024-02-01 02:00:00 |      24906.7 |
| 2024-02-01 03:00:00 |      25064.7 |
| ...

Now we can evaluate the model daily. Since the target is day-ahead forecasting, the evaluation has to mirror that pattern. The loop below walks day-by-day through a test window, adding each day's actual values to the training set before predicting the next day. We'll evaluate over the first few days of January as a sanity check.

In [12]:
test_dates = list(pd.date_range(start=datetime.datetime(2024, 1, 1), end=datetime.datetime(2024, 2, 1), freq='D'))

In [13]:
from darts.metrics import mape

mapes = []
for test_date in test_dates:
    train_df = pd.DataFrame(df.loc[:test_date-datetime.timedelta(minutes=1),])
    training = darts.TimeSeries.from_dataframe(train_df)
    test_df = pd.DataFrame(df[test_date:test_date+datetime.timedelta(days=1, minutes=-1)])
    test = darts.TimeSeries.from_dataframe(test_df)
    test_steps = len(test_df)
    result = train_forecast(training, test_steps)
    r = mape(test, result)
    mapes.append(r)

In [14]:
np.mean(mapes)

9.805162208019903

Now we can write the forecast back to InfluxDB as a new field on the existing series. Strategy:

- Focus on the most recent month of data. For the current (incomplete) day, fill in the remaining hours.
- Generate a day-ahead forecast for each day in the window.
- Write it back to InfluxDB.

Implementation in stages below.

---

### Task 3: Hourly timestamp generator

Implement a helper that, given a date, returns the sorted list of hourly timestamps for that day.

In [22]:
def get_hours(date: datetime) -> list:
    """
    Devuelve una lista con los timestamps horarios de la fecha dada.
    """
    return list(pd.date_range(start=date, periods=24, freq='H'))


In [23]:
get_hours(datetime(2023, 2, 28))

[Timestamp('2023-02-28 00:00:00', freq='H'),
 Timestamp('2023-02-28 01:00:00', freq='H'),
 Timestamp('2023-02-28 02:00:00', freq='H'),
 Timestamp('2023-02-28 03:00:00', freq='H'),
 Timestamp('2023-02-28 04:00:00', freq='H'),
 Timestamp('2023-02-28 05:00:00', freq='H'),
 Timestamp('2023-02-28 06:00:00', freq='H'),
 Timestamp('2023-02-28 07:00:00', freq='H'),
 Timestamp('2023-02-28 08:00:00', freq='H'),
 Timestamp('2023-02-28 09:00:00', freq='H'),
 Timestamp('2023-02-28 10:00:00', freq='H'),
 Timestamp('2023-02-28 11:00:00', freq='H'),
 Timestamp('2023-02-28 12:00:00', freq='H'),
 Timestamp('2023-02-28 13:00:00', freq='H'),
 Timestamp('2023-02-28 14:00:00', freq='H'),
 Timestamp('2023-02-28 15:00:00', freq='H'),
 Timestamp('2023-02-28 16:00:00', freq='H'),
 Timestamp('2023-02-28 17:00:00', freq='H'),
 Timestamp('2023-02-28 18:00:00', freq='H'),
 Timestamp('2023-02-28 19:00:00', freq='H'),
 Timestamp('2023-02-28 20:00:00', freq='H'),
 Timestamp('2023-02-28 21:00:00', freq='H'),
 Timestamp

Expected result:

```
[Timestamp('2023-02-28 00:00:00', freq='H'),
 Timestamp('2023-02-28 01:00:00', freq='H'),
 ...
 Timestamp('2023-02-28 23:00:00', freq='H')]
```

To write the forecast to InfluxDB we use a function similar to Part 1's write helper — it takes a Prophet forecast and writes it through InfluxDB's API. The forecast goes into the same `demand` measurement but under a new field `CP7Forecast`.

In [24]:
def save_to_influxdb(df: pd.DataFrame, client: InfluxDBClient, bucket: str, org: str) -> pd.DataFrame:
    """
    Escribe el pronóstico en InfluxDB
    """
    df = df.reset_index()
    df['time'] = df['datetime']
    to_write = df[['time', 'CP7Forecast']]
    to_write = to_write.set_index('time')[['CP7Forecast']]
    write_api = client.write_api(write_options=SYNCHRONOUS)
    write_api.write(bucket, org, record=to_write, data_frame_measurement_name='demand')

We now have everything needed to ingest the forecast. Starting from an initial date, we use the same rolling-window pattern as the evaluation, but instead of iterating over a test set we generate the hours ourselves. The code below ingests forecasts starting from a configurable date.

In [29]:
begin = datetime(2025, 5,1)
dates_to_ingest = pd.date_range(start=begin, end=datetime.now().date(), freq='D')
dates_to_ingest

DatetimeIndex(['2025-05-01', '2025-05-02', '2025-05-03', '2025-05-04',
               '2025-05-05', '2025-05-06', '2025-05-07', '2025-05-08',
               '2025-05-09', '2025-05-10', '2025-05-11', '2025-05-12',
               '2025-05-13', '2025-05-14', '2025-05-15', '2025-05-16',
               '2025-05-17', '2025-05-18', '2025-05-19', '2025-05-20',
               '2025-05-21'],
              dtype='datetime64[ns]', freq='D')

In [30]:
for dt in dates_to_ingest:
    hours = len(get_hours(dt))
    print(f"Ingesting for date: {dt}")
    train_df = pd.DataFrame(df[:dt-timedelta(minutes=1)])
    training = darts.TimeSeries.from_dataframe(train_df, freq='H', fill_missing_dates=True)
    result = train_forecast(training, hours).pd_dataframe()
    save_to_influxdb(result, client, bucket, org)

Ingesting for date: 2025-05-01 00:00:00
Ingesting for date: 2025-05-02 00:00:00
Ingesting for date: 2025-05-03 00:00:00
Ingesting for date: 2025-05-04 00:00:00
Ingesting for date: 2025-05-05 00:00:00
Ingesting for date: 2025-05-06 00:00:00
Ingesting for date: 2025-05-07 00:00:00
Ingesting for date: 2025-05-08 00:00:00
Ingesting for date: 2025-05-09 00:00:00
Ingesting for date: 2025-05-10 00:00:00
Ingesting for date: 2025-05-11 00:00:00
Ingesting for date: 2025-05-12 00:00:00
Ingesting for date: 2025-05-13 00:00:00
Ingesting for date: 2025-05-14 00:00:00
Ingesting for date: 2025-05-15 00:00:00
Ingesting for date: 2025-05-16 00:00:00
Ingesting for date: 2025-05-17 00:00:00
Ingesting for date: 2025-05-18 00:00:00
Ingesting for date: 2025-05-19 00:00:00
Ingesting for date: 2025-05-20 00:00:00
Ingesting for date: 2025-05-21 00:00:00


When the ingestion completes, your InfluxDB instance will have the forecast field populated. The last step is adapting the Part 1 dashboard to overlay the forecast on the existing real-vs-REE-forecast chart.

---

### Task 4: Dashboard update

Add the new `CP7Forecast` series to the Graph panel of the dashboard you built in Part 1.

To see the future forecast, you'll need to extend the time-range filter to include the current day.

Capture a screenshot of the updated dashboard.

Expected result (approximate):

![tarea4](img/p2tarea4.png)

My result:

![tarea4](img_luis/dashboard_2.png)

---